In [ ]:
# =============================================================================
# FUMD-AI Training & Postprocessing Workflow -- Step 3: Evaluate model
# =============================================================================
# Step:         3 of 5
# Summary:      Evaluate a trained run on its own held-out validation split, plus optional cross-run generalization checks.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-training-postprocessing-workflow
# Version:      0.1.3
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# =============================================================================

# Step 3 — Evaluate model

Part of the **FUMD-AI training & postprocessing workflow** (step 3 of 5).

**Purpose.** Evaluate a Step 2 run:

1. On its own held-out validation windows (`run_label="train_val"`):
   per-forecast-step precision/recall/F1/top-2-accuracy/confusion matrix
   (`evaluate.evaluate_per_output`), **plus** an accuracy breakdown by
   `migration` state (`evaluate.evaluate_migration_breakdown`) -- how good
   the model is specifically around real handovers, not just on average
   (steady-state rows dominate the dataset, so a trivial "predict no
   change" baseline already scores well there).
2. *(optional)* Against however many other `dataset_labeled_w<W>.csv` runs
   you list in `CROSS_RUN_DATASET_PATHS` -- a generalization check, reusing
   the *same* fitted encoders/scaler from Step 1 (transform-only, never
   re-fit -- see `data.apply_scaler`'s docstring for why re-fitting per
   dataset, as the original exploratory notebook did, makes results
   incomparable).

Every result is written as structured CSV + JSON
(`metrics_io.write_metrics`) instead of the original notebooks' printed-text
log + regex-scraping (`leer_metricas.ipynb`) -- Step 4 reads these directly.

**Input:** `RUN_DIR` -- a Step 1+2 output directory (`model.keras`,
`scaler.joblib`, `label_encoders.joblib`, `run_manifest.json`,
`windows.npz`).

**Outputs**, written under `METRICS_OUTPUT_DIR` (defaults to `RUN_DIR`):
`metrics_<run_label>.csv` / `.json` per evaluated dataset.


In [ ]:
import os
import sys

import joblib
import numpy as np

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # `jupyter execute` sets the kernel's cwd to this notebook's OWN
                     # directory (see nbclient's cli.py), not the repo root -- normalize
                     # back so `sys.path.insert(0, "src")` and every relative default
                     # below resolve the same way whether this notebook is run standalone
                     # (jupyter execute notebooks/step_N...ipynb, e.g. via slurm/*.sbatch)
                     # or chained by run_pipeline.ipynb (whose own cwd is already the repo
                     # root, and which additionally sets cwd explicitly -- see
                     # fumd_training_workflow.notebook_runner.run_step).

sys.path.insert(0, "src")
from fumd_training_workflow import data as d
from fumd_training_workflow.evaluate import evaluate_migration_breakdown, evaluate_per_output, summarize_overall
from fumd_training_workflow.metrics_io import load_run_manifest, write_metrics
from fumd_training_workflow.model import BahdanauAttention, ZeroInitialState

import tensorflow as tf

In [ ]:
# ---- Parameters ----
RUN_DIR = "pipeline_run"          # a Step 1+2 output directory
METRICS_OUTPUT_DIR = None          # None -> RUN_DIR

# Optional: other dataset_labeled_w<W>.csv paths (e.g. other preprocessing
# pipeline_run_*/ outputs) to check this model's generalization against.
# Left empty by default -- only the bundled example is guaranteed to exist.
CROSS_RUN_DATASET_PATHS = []

In [ ]:
# ---- Environment variable overrides (optional) ----
# `jupyter execute` has no -p/parameter-injection flag (unlike papermill), so
# each parameter above can also be set via an environment variable of the same
# name instead of hand-editing this cell, e.g.:
#   DATASET_PATH=/abs/path/dataset_labeled_w3.csv OUTPUT_DIR=/abs/path/pipeline_run \
#   jupyter execute notebooks/step_3_evaluate_model.ipynb
# See fumd_training_workflow.notebook_runner.env_override's docstring for exact
# value-parsing rules. Unset variables leave the parameters cell's own default above untouched.
from fumd_training_workflow.notebook_runner import env_override

RUN_DIR = env_override('RUN_DIR', RUN_DIR)
METRICS_OUTPUT_DIR = env_override('METRICS_OUTPUT_DIR', METRICS_OUTPUT_DIR)
CROSS_RUN_DATASET_PATHS = env_override('CROSS_RUN_DATASET_PATHS', CROSS_RUN_DATASET_PATHS)

## 1. Load the trained run

In [ ]:
METRICS_OUTPUT_DIR = METRICS_OUTPUT_DIR or RUN_DIR

manifest = load_run_manifest(RUN_DIR)
model = tf.keras.models.load_model(
    os.path.join(RUN_DIR, "model.keras"),
    custom_objects={"BahdanauAttention": BahdanauAttention, "ZeroInitialState": ZeroInitialState},
)
scaler = joblib.load(os.path.join(RUN_DIR, "scaler.joblib"))
label_encoders = joblib.load(os.path.join(RUN_DIR, "label_encoders.joblib"))

feature_columns = manifest["feature_columns"]
sequence_length = manifest["sequence_length"]
future_steps = manifest["future_steps"]
num_base_stations = manifest["num_base_stations"]

print(f"loaded run from {RUN_DIR}: trained on {manifest.get('dataset_path')}")

## 2. Evaluate on this run's own validation split

In [ ]:
data_npz = np.load(os.path.join(RUN_DIR, "windows.npz"))
X_val, y_val = data_npz["X_val"], data_npz["y_val"]
migration_val = data_npz["migration_val"]

y_pred_probs = model.predict(X_val)

per_output = evaluate_per_output(y_val, y_pred_probs, future_steps, num_base_stations)
overall = summarize_overall(per_output)
y_pred_first_step = np.asarray(y_pred_probs[0]).argmax(axis=-1)
migration_breakdown = evaluate_migration_breakdown(y_val[:, 0], y_pred_first_step, migration_val)

write_metrics(
    METRICS_OUTPUT_DIR, "train_val", per_output, overall, migration_breakdown,
    meta={"model_run": RUN_DIR, "dataset_path": manifest.get("dataset_path")},
)

print("overall (own validation split):", overall)
print("migration breakdown:", migration_breakdown)

## 3. Optional cross-run generalization check

Reuses this run's *fitted* label encoders and scaler (transform-only) --
never re-fit on the new dataset.

In [ ]:
results_summary = {"train_val": overall}

for dataset_path in CROSS_RUN_DATASET_PATHS:
    # Every preprocessing run produces a file with the SAME name
    # (dataset_labeled_w<W>.csv), so basing run_label on the filename alone
    # would collide across different runs and silently overwrite one
    # cross-run's metrics with another's. This project's own convention is
    # to distinguish runs by their containing directory (e.g. "900_1",
    # "pipeline_run_900_3") instead -- prefer that, falling back to the
    # filename only if the path has no parent directory to use.
    run_label = os.path.basename(os.path.dirname(os.path.abspath(dataset_path))) \
        or os.path.splitext(os.path.basename(dataset_path))[0]
    print(f"--- evaluating against {dataset_path} (run_label={run_label}) ---")

    df = d.load_labeled_dataset(dataset_path)
    df = d.ensure_one_row_per_vehicle_second(df)
    df = d.apply_label_encoders(df, label_encoders)  # transform-only; unseen lane ids map to the reserved "unknown" code

    X, y, extra = d.build_sequences(
        df, feature_columns=feature_columns, sequence_length=sequence_length, future_steps=future_steps,
    )
    X_scaled = d.apply_scaler(scaler, X)  # transform-only -- never re-fit (see data.apply_scaler)

    preds = model.predict(X_scaled)
    per_output_x = evaluate_per_output(y, preds, future_steps, num_base_stations)
    overall_x = summarize_overall(per_output_x)
    pred_first_step = np.asarray(preds[0]).argmax(axis=-1)
    breakdown_x = evaluate_migration_breakdown(y[:, 0], pred_first_step, extra["migration"])

    write_metrics(
        METRICS_OUTPUT_DIR, run_label, per_output_x, overall_x, breakdown_x,
        meta={"model_run": RUN_DIR, "dataset_path": dataset_path},
    )
    results_summary[run_label] = overall_x
    print(f"{run_label}: {overall_x}")

print()
print("summary (overall accuracy per evaluated dataset):")
for label, o in results_summary.items():
    print(f"  {label}: accuracy={o['accuracy']:.4f}, top2_accuracy={o['top2_accuracy']:.4f}")